# Build, Shape, and Extend an OpenClaw Agent on AMD
### From local agent basics to skills, customization, and a live adaptive app

---

**Your setup today:**
- AMD Instinct MI300X GPU (192 GB memory)
- Qwen3.5-122B model served via SGLang on `localhost:8090`
- OpenClaw agent framework
- A Python typing app with **3 real bugs** — your agent will find and fix them

Everything runs on this machine. Run cells in order.

---
## Section 0 — Verify Your Environment

The model server and OpenClaw are already installed. Let's confirm everything is up.

In [ ]:
import urllib.request, json, subprocess, time, pathlib, os, sys
sys.path.insert(0, ".")

# ── Verify SGLang is running ──────────────────────────────────────────────────
try:
    req = urllib.request.Request(
        "http://localhost:8090/v1/models",
        headers={"Authorization": "Bearer abc-123"}
    )
    with urllib.request.urlopen(req) as r:
        models = json.loads(r.read())
    print("✅ SGLang is running")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
except Exception as e:
    print(f"❌ SGLang not reachable: {e}")

In [ ]:
# ── Configure OpenClaw → SGLang ───────────────────────────────────────────────
BASE_URL = "http://localhost:8090/v1"
API_KEY  = "abc-123"
MODEL_ID = "qwen3-5-122b"

subprocess.run(["openclaw", "config", "set", "gateway.mode", "local"], check=True)
subprocess.run(["openclaw", "config", "set", "agents.defaults.model", f"sglang/{MODEL_ID}"], check=True)

config_path = pathlib.Path.home() / ".openclaw" / "openclaw.json"
with config_path.open() as f:
    cfg = json.load(f)

cfg.setdefault("models", {}).setdefault("providers", {})["sglang"] = {
    "baseUrl": BASE_URL, "apiKey": API_KEY, "api": "openai-completions",
    "models": [{"id": MODEL_ID, "name": MODEL_ID, "reasoning": True,
                "input": ["text"], "cost": {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
                "contextWindow": 131072, "maxTokens": 8192}]
}
with config_path.open("w") as f:
    json.dump(cfg, f, indent=2)
print("✅ OpenClaw configured")

In [ ]:
# ── Start the gateway ─────────────────────────────────────────────────────────
log_file = pathlib.Path("/tmp/openclaw-gateway.log")
subprocess.run(["pkill", "-9", "-f", "openclaw-gateway"], capture_output=True)
time.sleep(1)
proc = subprocess.Popen(
    ["openclaw", "gateway", "run", "--bind", "loopback", "--port", "18789", "--force"],
    stdout=log_file.open("w"), stderr=subprocess.STDOUT,
)
time.sleep(4)
if proc.poll() is None:
    print(f"✅ Gateway running (PID {proc.pid})")
else:
    print("❌ Gateway failed:\n", log_file.read_text()[-600:])

In [ ]:
# ── Smoke test ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ["openclaw", "chat", "Reply in one sentence: what model are you?"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

---
## Section 1 — How OpenClaw Is Structured

Before we touch any code, here is the mental model.

```
┌─────────────────────────────────────────────────────────────┐
│                        OpenClaw Stack                       │
│                                                             │
│  You ──► Gateway ──► Agent ──► Tools ──► World             │
│                         │                                   │
│                      Workspace                              │
│                   ┌─────┴──────┐                            │
│                 SOUL.md    USER.md        ← durable files   │
│                 AGENTS.md  SKILLS/        ← durable files   │
│                                                             │
│  Channels: WebChat · Slack · Teams · CLI                    │
│  ClawHub : shareable, installable skills                    │
└─────────────────────────────────────────────────────────────┘
```

| Concept | What it does |
|---|---|
| **Gateway** | Entry point — routes messages to the agent |
| **Agent** | Runs the loop: think → act → observe → repeat |
| **Workspace** | Where the agent lives and reads its context |
| **SOUL.md** | Defines the agent's personality and overall behavior |
| **USER.md** | Stores what the agent knows about you specifically |
| **Tools** | How the agent interacts with the world (files, shell, browser) |
| **Skills** | Reusable, packaged capabilities |

> **Key idea:** the agent's behavior is shaped by *durable files in the workspace*. Change the files, change the agent.

---
## Section 2 — The Broken App

This repo has **3 bugs** introduced across 3 different files.  
Users have been reporting:

- 🐛 **The cursor is always one character ahead** of where they're actually typing
- 🐛 **Net WPM looks wrong** — it's too high when sessions are short
- 🐛 **WPM tanks mysteriously** if they press backspace before starting to type

Let's run the tests and see exactly what's failing.

In [ ]:
# Run the test suite — this is the agent's starting point
result = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # show the end where failures appear

**4 tests failing across 3 files.**  
Now let's see what happens when we ask a plain `openclaw chat` to fix it —  
then compare that to a properly shaped agent with tools.

---
## Section 3 — Shape the Agent First

Before we let the agent loose on the bugs, we customize it.  
A generic agent will give generic answers. A shaped agent behaves like a good senior developer.

In [ ]:
# See the current SOUL.md — default behavior
workspace = subprocess.check_output(["openclaw", "workspace", "path"]).decode().strip()
soul_path = pathlib.Path(workspace) / "SOUL.md"
print(soul_path.read_text())

In [ ]:
# Ask a plain (unshaped) agent to suggest what's wrong
# It has no repo context — watch how vague this is
result = subprocess.run(
    ["openclaw", "chat",
     "A typing app has bugs: cursor appears one char too far right, net WPM is too high on short sessions, "
     "and WPM tanks when backspace is pressed early. What might be wrong?"],
    capture_output=True, text=True
)
print("── Unshaped agent ──")
print(result.stdout)

In [ ]:
# Now shape the SOUL — turn it into a senior developer persona
dev_soul = """

## Code Debugging Mode
You are a careful senior developer debugging a Python codebase. When given a bug:
1. Read the failing tests first — they describe the expected behaviour exactly
2. Trace from the test failure back to the source, file by file
3. State the root cause clearly before suggesting any fix
4. Make the minimal change needed — do not refactor unrelated code
5. After fixing, explain what you changed and why the original code was wrong
"""

with soul_path.open("a") as f:
    f.write(dev_soul)
print("✅ SOUL.md updated")

---
## Section 4 — Give the Agent the Task

Now we hand the agent the repo and tell it to investigate.  
Watch what it does differently from the plain chat above.

In [ ]:
# Point the agent at the repo directory
repo_path = pathlib.Path(".").resolve()
print(f"Repo path: {repo_path}")
print("Files the agent can read:")
for f in sorted(repo_path.glob("*.py")):
    print(f"  {f.name}")

In [ ]:
# Give the agent a real task with the repo as context
# openclaw run allows the agent to use tools: read files, run commands, write edits
task = f"""I have a Python typing speed app at {repo_path}.
The test suite is failing. Run the tests, read the relevant source files,
identify the root cause of each failure, and fix them.
Start by running: python3 -m pytest tests/ -v --tb=short"""

print("Task sent to agent:")
print(task)

In [ ]:
# Run the agent — this is the demo moment
# The agent will: run tests → read files → identify bugs → write fixes → verify
result = subprocess.run(
    ["openclaw", "run", task],
    capture_output=True, text=True, cwd=str(repo_path)
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

---
## Section 5 — Verify the Fixes

Did the agent actually fix the bugs? Let's run the tests and check.

In [ ]:
result = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-v"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])

passed = "failed" not in result.stdout
print("\n" + ("✅ All tests passing — bugs fixed!" if passed else "❌ Some tests still failing"))

In [ ]:
# Show the actual diffs the agent made
diff = subprocess.run(["git", "diff"], capture_output=True, text=True, cwd=str(repo_path))
print(diff.stdout)

Three bugs, three files, found and fixed autonomously.  
The agent ran the tests, read the code, reasoned about root causes, and wrote targeted fixes.

**This is what agents are for** — not answering questions, but doing developer work.

---
## Section 6 — Package It as a Reusable Skill

What the agent just did was ad-hoc. A **skill** makes this capability reusable —  
you can invoke it on any Python repo, not just this one.

In [ ]:
skill_dir = pathlib.Path(workspace) / "skills" / "pytest-debugger"
skill_dir.mkdir(parents=True, exist_ok=True)

skill_md = """# pytest-debugger

## Purpose
Investigate a failing Python test suite, identify the root cause of each
failure, and apply minimal fixes to make all tests pass.

## When to use
Invoke when given a repo path and told that tests are failing.

## Steps
1. Run `python3 -m pytest <path>/tests/ -v --tb=short` and capture output
2. For each FAILED test: read the test to understand expected behaviour
3. Read the source file referenced in the traceback
4. Identify the minimal line(s) causing the failure
5. Apply the fix — change only what is necessary
6. Re-run the tests to confirm the fix works
7. Report: file changed, line changed, what was wrong, what the fix was

## Rules
- Never rewrite functions — make the smallest possible edit
- Never skip a test — fix the code, not the test
- Always re-run after fixing to confirm before moving to the next bug
"""

(skill_dir / "skill.md").write_text(skill_md)
print("✅ pytest-debugger skill created")
print(skill_md)

In [ ]:
# Confirm OpenClaw sees it
!openclaw skills list

Now anyone on your team (or from ClawHub) can run:
```
openclaw skill pytest-debugger /path/to/any/python/repo
```
and get the same systematic debugging behaviour — shaped by the same SOUL, guided by the skill.

---
## Section 7 — ClawHub

What you built today is local. ClawHub makes it shareable.

```
Local skill  →  openclaw skill publish  →  ClawHub registry
Anyone       →  openclaw skill install pytest-debugger  →  their workspace
```

The skill carries the SOUL context with it — whoever installs it gets the same
careful debugging behaviour you just shaped, not a generic agent.

---
## Section 8 — Your Challenge

### Toolsmith Challenge

Your repo has a new feature branch with a broken feature. Use your shaped agent to find and fix it.

**Tracks — pick one:**

| Track | What to do |
|---|---|
| **Bug Hunter** | Introduce your own bug into a copy of the repo, swap with a neighbour, fix each other's bugs using the agent |
| **Skill Builder** | Extend `pytest-debugger` — add a step that writes a short bug report (file, line, root cause, fix) to `bug_report.md` |
| **Soul Shaper** | Change SOUL.md to make the agent explain bugs as if teaching a junior developer, re-run the same task, compare the output |

**What you have:**
- A working agent connected to Qwen3.5-122B on MI300X
- A SOUL shaped for careful developer behaviour
- A `pytest-debugger` skill as a template
- A real Python codebase with tests

---

*Built something interesting? Share it publicly and you may qualify for additional AMD Developer Cloud credits.*